[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_04/building_medical_dataset.ipynb)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install Wikipedia-API vllm synthetic-data-kit

### Listing 4.15: Functions to Retrieve Data from Wikipedia

In [ ]:
import re
from tqdm import tqdm ### FIX
import wikipediaapi

def clean_string(input_string):
    cleaned_string = " ".join(input_string.split())
    cleaned_string = re.sub(r"\{.*?\}|\}", "", cleaned_string)
    return cleaned_string

def get_wikipedia_content(titles):
    wiki = wikipediaapi.Wikipedia("MedicalBot/1.0 (medbot@example.com)", "en")
    extracted_texts = []
    print("- Fetching medical articles:")
    for title in tqdm(titles):
        page = wiki.page(title)
        if page.exists():
            extracted_texts.append(page.title + " : " + clean_string(page.summary))
            for section in page.sections:
                if len(section.text) > 200:
                    extracted_texts.append(
                        page.title
                        + " ("
                        + section.title
                        + ") : "
                        + clean_string(section.text)
                    )
    return extracted_texts


### Listing 4.16: List of Cardiology Topics

In [ ]:
cardiology_topics = [
    "Hypertension", "Acute coronary syndrome", "Myocardial infarction", 
    "Stable angina", "Unstable angina", "Coronary artery disease", 
    "Coronary artery spasm", "TIMI risk score", "Percutaneous coronary intervention",
    "Coronary artery bypass surgery", "Heart failure",
    "Heart failure with preserved ejection fraction", "Dilated cardiomyopathy",
    "Hypertrophic cardiomyopathy", "Restrictive cardiomyopathy",
    "Takotsubo cardiomyopathy", "Cardiac resynchronization therapy",
    "Atrial fibrillation", "Atrial flutter", "Supraventricular tachycardia",
    "Ventricular tachycardia", "Ventricular fibrillation",
    "Wolff-Parkinson-White syndrome", "Long QT syndrome", "Brugada syndrome",
    "Sick sinus syndrome", "Atrioventricular block", "Cardiac pacemaker",
    "Implantable cardioverter-defibrillator", "Aortic stenosis",
    "Aortic regurgitation", "Mitral stenosis", "Mitral regurgitation",
    "Mitral valve prolapse", "Tricuspid regurgitation", "Infective endocarditis",
    "Transcatheter aortic valve replacement",
    "Pericarditis", "Cardiac tamponade", "Beck's triad (cardiac tamponade)",
    "Myocarditis", "Constrictive pericarditis", "Aortic aneurysm",
    "Aortic dissection", "Peripheral artery disease",
    "Deep vein thrombosis", "Pulmonary embolism", "Dyslipidemia",
    "Familial hypercholesterolemia", "Atherosclerosis", "Metabolic syndrome",
    "Congenital heart disease", "Atrial septal defect", "Ventricular septal defect", 
    "Tetralogy of Fallot", "Patent ductus arteriosus", "Pulmonary hypertension",
    "Cor pulmonale", "Electrocardiography", "Echocardiography", "Cardiac stress test",
    "Cardiac catheterization", "Holter monitor", "Troponin", 
    "B-type natriuretic peptide", "Beta blocker", "ACE inhibitor", 
    "Calcium channel blocker", "Statin", "Anticoagulant", "Antiarrhythmic agent",
    "Diuretic", "Digoxin", "Nitrate (pharmacology)",
]


### Listing 4.17: Extraction of Topics from Wikipedia

In [ ]:
import os

os.makedirs("data/medical_input", exist_ok=True)

extracted_texts = get_wikipedia_content(cardiology_topics)
for i, text in enumerate(extracted_texts):
    with open(os.path.join("data/medical_input", f"med_{i}.txt"), "w") as f:
        f.write(text)

### Listing 4.18: Starting a vLLM Server

In [ ]:
import time
import subprocess

MODEL_ENGINE = "Qwen/Qwen2.5-7B-Instruct-AWQ"

def start_vllm():
    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["VLLM_MOE_KERNEL_BACKEND"] = "triton"
    env["VLLM_DISABLED_KERNELS"] = "cutlass_moe_mm,cutlass_scaled_mm"

    vllm_log = open("vllm_server.log", "w")
    return subprocess.Popen(
        [
            "vllm",
            "serve",
            MODEL_ENGINE,
            "--port", "8000",
            "--enforce-eager",
            "--quantization", "awq",
            "--gpu-memory-utilization", "0.80",
            "--max-model-len", "8192",
            "--disable-log-stats",
            "--uvicorn-log-level", "error",
        ],
        env=env,
        stdout=vllm_log,
        stderr=vllm_log,
    )

vllm_proc = start_vllm()

print("Waiting for vLLM server to be ready...")
while "Starting vLLM server" not in open("vllm_server.log").read():
    time.sleep(2)
print("vLLM server ready!")

### Listing 4.19: Initiating the vLLM Configuration YAML File

In [ ]:
yaml_content = f"""

paths:
  input:
    txt: "data/medical_input"
  output:
    generated: "data/generated"
    curated: "data/curated"
    final: "data/final"

vllm:
  api_base: "http://localhost:8000/v1"
  port: 8000
  model: "{MODEL_ENGINE}"
  max_retries: 3
  retry_delay: 1.0
"""

### Listing 4.20: Adding further Configurations to the vLLM Configuration YAML File

In [ ]:
NUM_PAIRS_PER_FILE = 30
QUALITY_THRESHOLD = 7.0

yaml_content += f"""
  generation:
  temperature: 0.4
  chunk_size: 1022
  overlap: 64
  num_pairs: {NUM_PAIRS_PER_FILE}

curate:
  threshold: {QUALITY_THRESHOLD}
  batch_size: 4
  temperature: 0.3
"""

### Listing 4.21: Completing the vLLM Configuration YAML File

In [ ]:
yaml_content += f"""
prompts:
  summary: |
    Summarize the following medical text in 3-5 concise sentences.
    Focus on the key clinical concepts, diagnostic criteria, and treatment highlights.
    Return ONLY the summary text, no preamble.
    Text: {{text}}

  qa_generation: |
    Create {NUM_PAIRS_PER_FILE} question-answer pairs from this medical text for clinical training.
    Focus on diagnostic criteria, treatments, and pathophysiology.
    Output ONLY a valid JSON array, no explanations, no markdown:

    [
      {{
        "question": "Question 1?",
        "answer": "Answer 1."
      }},
      {{
        "question": "Question 2?",
        "answer": "Answer 2."
      }}
    ]

    Text: {{text}}

  qa_rating: |
    Rate each of these question-answer pairs for quality and return exactly this JSON format:

    [
      {{"question": "same question text", "answer": "same answer text", "rating": n}}
    ]

    Where n is a number from 1-10, using this scale:
    1=wrong/nonsensical
    3=vague/incomplete
    5=correct but generic
    7=accurate and useful
    9=insightful and precise
    10=perfect

    DO NOT include any text outside of the JSON array, just return valid JSON:

    {{pairs}}
"""

with open("medical_data_config.yaml", "w") as f:
    f.write(yaml_content)

### Listing 4.22: Starting the Creation Process for Q/A Synthetic Data Generation

In [ ]:
input_files = [
    os.path.join("data/medical_input", f)
    for f in os.listdir("data/medical_input")
    if f.endswith(".txt")
]
vllm_log = open("vllm_server.log", "a")

print("Starting generation and curation loop...")
for filepath in tqdm(input_files):
    subprocess.run(
        [
            "synthetic-data-kit",
            "-c",
            "medical_data_config.yaml",
            "create",
            filepath,
            "--type",
            "qa",
        ],
        stdout=vllm_log,
        stderr=vllm_log,
    )

 95%|█████████▌| 510/536 [25:31<01:18,  3.00s/it]

### Listing 4.23: Evaluation Process for Q/A Synthetic Data Generation

In [ ]:
for filepath in tqdm(input_files):
    gen_file = os.path.join(
        "data/generated", os.path.basename(filepath).replace(".txt", "_qa_pairs.json")
    )
    if os.path.exists(gen_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "curate",
                gen_file,
            ],
            stdout=vllm_log,
            stderr=vllm_log,
        )

### Listing 4.24: Curating the Results from the Q/A Synthetic Data Generation

In [ ]:
for filepath in tqdm(input_files):
    curated_file = os.path.join(
        "data/curated",
        os.path.basename(filepath).replace(".txt", "_qa_pairs_cleaned.json"),
    )
    if os.path.exists(curated_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "save-as",
                curated_file,
                "-f",
                "ft",
            ],
            stdout=vllm_log,
            stderr=vllm_log,
        )

vllm_proc.terminate()
vllm_log.close()

### Listing 4.25: Wrapping the Results from the Q/A Synthetic Data Generation into a Dataset

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

SYSTEM_PROMPT = "You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology."

def set_system_prompt(row):
    row["messages"][0]["content"] = SYSTEM_PROMPT
    return row

conversations = pd.concat(
    [pd.read_json(f"data/final/{name}") for name in os.listdir("data/final")]
).reset_index(drop=True)

conversations = conversations.apply(set_system_prompt, axis=1)

complete_dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(conversations),
    }
)

In [ ]:
complete_dataset.save_to_disk("data/medical-cardiology-qa")

In [ ]:
for i in range(3): 
    print(f"Q: {complete_dataset['train'][i]['messages'][1]['content']}")
    print(f"A: {complete_dataset['train'][i]['messages'][2]['content']}\n")